<a href="https://colab.research.google.com/github/geeta-gwalior/Gemma-with-RAG-A-Coffee-Shop-Chatbot/blob/main/Gemma_with_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


!pip install -U "transformers>=4.50.0" accelerate torch safetensors bitsandbytes
!pip install -U sentence-transformers faiss-cpu


In [ ]:
from huggingface_hub import notebook_login
import os
from google.colab import userdata

from huggingface_hub import login

login(token=userdata.get('hf_token'))
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-3-1b-it"  # or gemma-7b-it if GPU allows

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

inputs = tokenizer("Tell me something about espresso coffee.", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:

import torch



tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

chat_history = []

def chat(query):
    global chat_history
    chat_history.append({"role":"user","content":query})
    prompt = tok.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
    reply = tok.decode(outputs[0], skip_special_tokens=True)

    # extract only assistant's last message
    reply = reply.split(query)[-1].strip()
    chat_history.append({"role":"assistant","content":reply})
    return reply


In [ ]:
while True:
    user_inp = input("You: ")
    if user_inp.lower() in ["quit","exit","stop"]:
        break
    response = chat(user_inp)
    print("Gemma:", response)


In [ ]:
import pandas as pd

# Load your uploaded JSONL dataset
df = pd.read_json("/content/coffeeworld_dataset.jsonl", lines=True)
display(df.head())

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

# Use a small fast embedding model
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Store the input questions and outputs
questions = df["input"].tolist()
answers = df["output"].tolist()

# Encode the input questions
X = embedder.encode(questions, normalize_embeddings=True)

# Build FAISS index
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)


In [ ]:
chat_history = []

def coffee_chat(query, k=1):
    global chat_history

    # Find best-matching FAQ question
    qv = embedder.encode([query], normalize_embeddings=True)
    D, I = index.search(qv, k)

    # Retrieve the most relevant answer
    answer = answers[I[0][0]]

    # Add to history
    chat_history.append({"user": query, "bot": answer})

    return answer


In [ ]:
while True:
    user_inp = input("You: ")
    if user_inp.lower() in ["quit","exit","stop"]:
        print("CoffeeBot: Goodbye 👋")
        break
    response = coffee_chat(user_inp)
    print("CoffeeBot:", response)
